# Normal modes and beating

Install the project into the selected kernel and run the cells in order.
Use metres, kilograms, and seconds. Start with identical masses, a wall
spring on each side, and a coupling spring between the masses.

## 1. Predict the frequencies

In phase, the coupling spring does not stretch, giving
$\omega_1=\sqrt{k/m}$. Out of phase, it adds twice the coupling stiffness,
giving $\omega_2=\sqrt{(k+2\kappa)/m}$. Choose the ratio $21/20$ so the two
cosines are simultaneously at extrema at the first exchange time.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from modal_lab import assemble, describe_modes, energy, normal_modes, simulate
from modal_lab import spring_mass_chain, symmetric_two_dof
from modal_lab.plotting import plot_beating, plot_modes

m, k = 1.0, 1.0
coupling = k * ((21 / 20)**2 - 1) / 2
system = spring_mass_chain(mass=m, wall=k, coupling=coupling)
low, high, beat = symmetric_two_dof(m, k, coupling)
print(f"Theory: ω = {low:.8f}, {high:.8f} rad/s, Tbeat = {beat:.4f} s")

## 2. Solve and interpret the eigenproblem

Solve $Kv=\omega^2Mv$ and check $V^TMV=I$. The node at 1.5 in the
out-of-phase mode is an interpolated zero crossing between masses.

In [ ]:
modes = normal_modes(system)
mass, _, stiffness = assemble(system)
describe_modes(modes)
print("Mass-weighted inner products:")
print(modes.shapes.T @ mass @ modes.shapes)
np.testing.assert_allclose(modes.omega, [low, high], atol=1e-10, rtol=0)
plot_modes(modes)
plt.show()

## 3. Watch energy move between oscillators

Release mass one from rest with unit displacement. Superposing the
two modes gives $q_{1,2}=(\cos\omega_1t\pm\cos\omega_2t)/2$.
At $T_b/2$ the initial displacement and zero velocity swap exactly.
The second mass now holds all the energy associated with its grounded
oscillator, and the coupling spring still stores energy. The plots share
this coupling energy equally between the masses.

In [ ]:
time = np.linspace(0, beat, 2401)
run = simulate(system, time, q0=[1, 0])
parts = energy(system, run)
exact = np.column_stack((np.cos(low * time) + np.cos(high * time),
                         np.cos(low * time) - np.cos(high * time))) / 2
mid = len(time) // 2
print(f"Exchange coordinates: {run.displacement[mid]}")
print(f"Energy fraction assigned to mass two: {parts.local[mid, 1] / parts.total[mid]:.3%}")
print(f"Maximum trajectory error: {np.max(abs(run.displacement - exact)):.2e} m")
plot_beating(run, parts)
plt.show()

Repeat with a smaller coupling to see the energy localization improve
and the beat period grow. Try a free-free chain and identify its rigid mode.